In [3]:
!pip install stanza pandas -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import stanza
import pandas as pd
import re
from IPython.display import display

In [5]:
stanza.download('uk', processors='tokenize,pos,lemma')
nlp = stanza.Pipeline(lang='uk', processors='tokenize,pos,lemma', use_gpu=False)

2026-03-03 21:05:00 INFO: Downloaded file to C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\resources.json
2026-03-03 21:05:00 WARNING: Language uk package default expects mwt, which has been added
2026-03-03 21:05:00 INFO: Downloading these customized packages for language: uk (Ukrainian)...
| Processor       | Package     |
---------------------------------
| tokenize        | iu          |
| mwt             | iu          |
| pos             | iu_charlm   |
| lemma           | iu_nocharlm |
| pretrain        | conll17     |
| forward_charlm  | conll17     |
| backward_charlm | conll17     |

2026-03-03 21:05:00 INFO: File exists: C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\uk\tokenize\iu.pt
2026-03-03 21:05:00 INFO: File exists: C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\uk\mwt\iu.pt
2026-03-03 21:05:01 INFO: File exists: C:\Users\mfese\AppData\Local\StanfordNLP\stanza\Cache\1.11.0\resources\uk\pos\iu_char

In [6]:
gold_subset = [
    {"raw_text": "Шукаємо крутого Senior Data Engineer з досвідом 3+ років.", "expected_exp": "3+"},
    {"raw_text": "Вимоги: комерційний досвід роботи від 1,5 року.", "expected_exp": "1,5"},
    {"raw_text": "Розглядаємо кандидатів без досвіду роботи.", "expected_exp": None},
    {"raw_text": "Бажаний досвід 2 роки в IT.", "expected_exp": "2"},
    {"raw_text": "Компанії вже 10 років на ринку, шукаємо мідла з досвідом 5 років.", "expected_exp": "5"}
]

EXPERIENCE_RULE = re.compile(
    r"(?:досвід[уа]?|experience|exp)[\s\w]{0,20}?(\d+(?:[.,]\d+)?\s*(?:\+|-)?\s*(?:\d+)?(?:-х)?)\s*(?:рок|років|years|yrs|р\.)",
    re.IGNORECASE
)

In [7]:
def get_lemma_text(text):
    doc = nlp(text)
    lemmas = []
    for sentence in doc.sentences:
        for word in sentence.words:
            lemmas.append(word.lemma if word.lemma else word.text)
    return " ".join(lemmas)

In [8]:
for row in gold_subset:
    row["lemma_text"] = get_lemma_text(row["raw_text"])

print(f"raw: {gold_subset[0]['raw_text']}")
print(f"lemma: {gold_subset[0]['lemma_text']}")

raw: Шукаємо крутого Senior Data Engineer з досвідом 3+ років.
lemma: шукати крутий Senior Data Engineer з досвід 3+ рік .


In [9]:
res = []

for row in gold_subset:
    match_raw = EXPERIENCE_RULE.search(row["raw_text"])
    pred_raw = match_raw.group(1).strip() if match_raw else None
    match_lemma = EXPERIENCE_RULE.search(row["lemma_text"])
    pred_lemma = match_lemma.group(1).strip() if match_lemma else None
    
    res.append({
        "Текст (Raw)": row["raw_text"],
        "Expected": row["expected_exp"],
        "Pred (Raw Baseline)": pred_raw,
        "Pred (Lemma Baseline)": pred_lemma
    })

df_res = pd.DataFrame(res)
display(df_res)

correct_raw = sum(1 for r in res if r["Expected"] == r["Pred (Raw Baseline)"])
correct_lemma = sum(1 for r in res if r["Expected"] == r["Pred (Lemma Baseline)"])

print(f"Baseline 1 (raw): {correct_raw} / {len(gold_subset)}")
print(f"Baseline 2 (lemma): {correct_lemma} / {len(gold_subset)}")

,Текст (Raw),Expected,Pred (Raw Baseline),Pred (Lemma Baseline)
0,Шукаємо крутого Senior Data Engineer з досвідо...,3+,3+,None
1,"Вимоги: комерційний досвід роботи від 1,5 року.","1,5","1,5",None
2,Розглядаємо кандидатів без досвіду роботи.,NaN,NaN,None
3,Бажаний досвід 2 роки в IT.,2,2,None
4,"Компанії вже 10 років на ринку, шукаємо мідла ...",5,5,None


Baseline 1 (raw): 5 / 5
Baseline 2 (lemma): 1 / 5
